In [ ]:
import pandas as pd
import numpyro
from jax.random import PRNGKey
from numpyro import distributions as dist
from numpyro import infer
import matplotlib.pyplot as plt
import diffrax as dfx

from tb_macro.constants import AGE_STRATA, ISO3, START_TIME, END_TIME
from tb_macro.epi import get_base_model, add_flows_to_model, initialise_pops
from tb_macro.inputs import load_demography, load_fertility, load_who_outcomes
from tb_macro.parameters import BASE_PARAMS
from tb_macro.demography import prepare_pop_data_for_entries
from tb_macro.calibration import make_log_likelihood, get_runner
from tb_macro.targets import NOTIF_TARGET, LATENT_TARGET

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [ ]:
# Model construction
group_popsize, death_rates, age_weights = load_demography(ISO3)
fert_padded = load_fertility(ISO3)
tsr, death_in_unsucc, who_mort = load_who_outcomes(ISO3)
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))
add_flows_to_model(
    epi_model, 
    disease_state,
    age_strat,
    clin_strat,
    infect_strat,
    age_weights,
    group_popsize,
    fert_padded,
    death_rates,
    tsr,
    death_in_unsucc,
    entry_times,
    entry_rates,
)
initialise_pops(epi_model, disease_state, age_strat, start_apops)

In [ ]:
runner, istate = get_runner(epi_model)

In [ ]:
solver_kwargs = {
    "max_steps": 4000,
    "stepsize_controller": dfx.PIDController(rtol=1e-5, atol=1e-5, dtmax=7.0),
    "adjoint": dfx.RecursiveCheckpointAdjoint(2048),
}

In [ ]:
latent_date = LATENT_TARGET.index[0]
latent_target_val = LATENT_TARGET.iloc[0] / 1e2

calib_params = ["contact_rate", "detect_val_2"]

log_like = make_log_likelihood(epi_model, disease_state, solver_kwargs, latent_date, latent_target_val, NOTIF_TARGET, who_mort)

In [ ]:
# Calibration
priors = {
    "contact_rate": dist.Uniform(5.0, 11.0),
    "detect_val_2": dist.Uniform(0.4, 0.7),
}


def model():
    params = BASE_PARAMS | {k: numpyro.sample(k, v) for k, v in priors.items()}
    ll = log_like(params)
    numpyro.factor("ll", ll)


# kernel = infer.SA(model)  # , adapt_state_size=4)  # infer.NUTS(model, max_tree_depth=5)
kernel = infer.NUTS(model, max_tree_depth=5, init_strategy=infer.init_to_median())
mcmc = infer.MCMC(kernel, num_warmup=100, num_samples=100, num_chains=4)
mcmc.run(PRNGKey(2))